# CardioAgent-Ops — QLoRA Qwen2.5-7B

Notebook d'exécution **Google Colab** (GPU T4 16 Go ou L4 24 Go).

> ⚠️ Projet académique. Le modèle n'est pas un dispositif médical et ne doit pas être utilisé
> pour un diagnostic.

1. Cloner le dépôt dans `/content` (ou monter Google Drive).
2. Exécuter les cellules dans l'ordre.
3. Télécharger `llm_finetuning/outputs/`, `llm_finetuning/export/` et `llm_finetuning/mlruns.db`.

In [1]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), 'Go')

Tue Nov 24 10:14:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2050      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   43C    P0              4W /   58W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Adapter le chemin si le dépôt est ailleurs.
%cd /content/cadrioagent-ops
!pip install -U pip -q
!pip install -r llm_finetuning/requirements-llm.txt -q

[WinError 3] Le chemin d’accès spécifié est introuvable: '/content/cadrioagent-ops'
c:\Users\Admin\Desktop\Semester_1_3GMAM\Data Mining\llm_finetuning\notebooks


c:\Users\Admin\Desktop\Semester_1_3GMAM\Data Mining\.venv\lib\site-packages\IPython\core\magics\osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'llm_finetuning/requirements-llm.txt'


In [3]:
!python llm_finetuning/data/build_dataset.py
!python llm_finetuning/data/build_dataset.py --check
!python -m pytest llm_finetuning/tests -q

C:\Users\Admin\AppData\Local\Programs\Python\Python310\python.exe: can't open file 'c:\\Users\\Admin\\Desktop\\Semester_1_3GMAM\\Data Mining\\llm_finetuning\\notebooks\\llm_finetuning\\data\\build_dataset.py': [Errno 2] No such file or directory
C:\Users\Admin\AppData\Local\Programs\Python\Python310\python.exe: can't open file 'c:\\Users\\Admin\\Desktop\\Semester_1_3GMAM\\Data Mining\\llm_finetuning\\notebooks\\llm_finetuning\\data\\build_dataset.py': [Errno 2] No such file or directory



no tests ran in 0.00s


ERROR: file or directory not found: llm_finetuning/tests



In [4]:
!python -m llm_finetuning.src.train --config llm_finetuning/configs/qlora_qwen25_7b.yaml --dry-run

c:\Users\Admin\Desktop\Semester_1_3GMAM\Data Mining\.venv\Scripts\python.exe: Error while finding module specification for 'llm_finetuning.src.train' (ModuleNotFoundError: No module named 'llm_finetuning')


In [5]:
# Entraînement complet (2 époques). Réduire max_steps pour un smoke test.
!python -m llm_finetuning.src.train --config llm_finetuning/configs/qlora_qwen25_7b.yaml

c:\Users\Admin\Desktop\Semester_1_3GMAM\Data Mining\.venv\Scripts\python.exe: Error while finding module specification for 'llm_finetuning.src.train' (ModuleNotFoundError: No module named 'llm_finetuning')


In [6]:
!python -m llm_finetuning.src.evaluate \
  --config llm_finetuning/configs/qlora_qwen25_7b.yaml \
  --adapter llm_finetuning/outputs/qwen25-7b-cardio/adapter \
  --compare-base

c:\Users\Admin\Desktop\Semester_1_3GMAM\Data Mining\.venv\Scripts\python.exe: Error while finding module specification for 'llm_finetuning.src.evaluate' (ModuleNotFoundError: No module named 'llm_finetuning')


In [7]:
!git clone https://github.com/ggerganov/llama.cpp llm_finetuning/export/llama.cpp
!cmake -S llm_finetuning/export/llama.cpp -B llm_finetuning/export/llama.cpp/build
!cmake --build llm_finetuning/export/llama.cpp/build --config Release -j
!python -m llm_finetuning.src.export_gguf \
  --config llm_finetuning/configs/qlora_qwen25_7b.yaml \
  --adapter llm_finetuning/outputs/qwen25-7b-cardio/adapter \
  --output-dir llm_finetuning/export --quantization q4_k_m

Cloning into 'llm_finetuning/export/llama.cpp'...
error: RPC failed; curl 56 schannel: server closed abruptly (missing close_notify)
error: 5799 bytes of body are still expected
fetch-pack: unexpected disconnect while reading sideband packet
fatal: early EOF
fatal: fetch-pack: invalid index-pack output
'cmake' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.
'cmake' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.
c:\Users\Admin\Desktop\Semester_1_3GMAM\Data Mining\.venv\Scripts\python.exe: Error while finding module specification for 'llm_finetuning.src.export_gguf' (ModuleNotFoundError: No module named 'llm_finetuning.src')


## Récupération des artefacts

- Le GGUF et le `Modelfile` sont dans `llm_finetuning/export/`.
- L'adaptateur est dans `llm_finetuning/outputs/qwen25-7b-cardio/adapter/`.
- La base MLflow locale est `llm_finetuning/mlruns.db` (à ouvrir avec
  `mlflow ui --backend-store-uri sqlite:///llm_finetuning/mlruns.db`).

Sur la machine locale :

```bash
ollama create cardio-llm -f llm_finetuning/export/Modelfile
ollama run cardio-llm
```

> L'intégration Docker (`llm-server`) se fera après validation du fine-tuning.